### 194. Fix the feature window convention first: every feature for a prediction at t+1 is computed from the trailing window ending at `t`, inclusive of `t` and nothing after it. Write this down before writing code.

For every prediction at `t+1`, all features are calculated using the trailing window ending at `t`, including `t` itself. No data after `t` is used.

This ensures that future information does not leak into the features.


### 195. Create avg_activity over the recent trailing window.

`avg_activity` is the average `total_activity` over the recent 24-hour trailing window, including time `t`.

It represents the grid's recent activity level and uses no data after `t`.


In [1]:
# 195. Create avg_activity over the recent trailing window.

import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="nopis"
)

query = """
SELECT
    f.grid_id,
    d.timestamp AS feature_timestamp,
    f.total_activity
FROM fact_network_activity f
JOIN dim_time d
    ON f.time_key = d.time_key
ORDER BY f.grid_id, d.timestamp;
"""

df = pd.read_sql(query, conn)

df.head()

D:\NOPIS\tmp\ipykernel_23208\1081132044.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,grid_id,feature_timestamp,total_activity
0,1,2013-10-31 18:30:00,61.6037
1,1,2013-10-31 19:30:00,46.1967
2,1,2013-10-31 20:30:00,42.0324
3,1,2013-10-31 21:30:00,35.0705
4,1,2013-10-31 22:30:00,32.2687


In [2]:
# 195. Create avg_activity over the recent trailing window.

df["feature_timestamp"] = pd.to_datetime(df["feature_timestamp"])

df["avg_activity"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.rolling(window=24, min_periods=24).mean())
)

df.head(30)

,grid_id,feature_timestamp,total_activity,avg_activity
0,1,2013-10-31 18:30:00,61.6037,NaN
1,1,2013-10-31 19:30:00,46.1967,NaN
2,1,2013-10-31 20:30:00,42.0324,NaN
3,1,2013-10-31 21:30:00,35.0705,NaN
4,1,2013-10-31 22:30:00,32.2687,NaN
5,1,2013-10-31 23:30:00,35.2842,NaN
6,1,2013-11-01 00:30:00,35.9625,NaN
7,1,2013-11-01 01:30:00,44.4976,NaN
8,1,2013-11-01 02:30:00,65.8393,NaN
9,1,2013-11-01 03:30:00,82.0947,NaN


In [3]:
# 195. Verify avg_activity uses exactly 24 observations.

df[df["grid_id"] == 1][
    ["grid_id", "feature_timestamp", "total_activity", "avg_activity"]
].head(30)

,grid_id,feature_timestamp,total_activity,avg_activity
0,1,2013-10-31 18:30:00,61.6037,NaN
1,1,2013-10-31 19:30:00,46.1967,NaN
2,1,2013-10-31 20:30:00,42.0324,NaN
3,1,2013-10-31 21:30:00,35.0705,NaN
4,1,2013-10-31 22:30:00,32.2687,NaN
5,1,2013-10-31 23:30:00,35.2842,NaN
6,1,2013-11-01 00:30:00,35.9625,NaN
7,1,2013-11-01 01:30:00,44.4976,NaN
8,1,2013-11-01 02:30:00,65.8393,NaN
9,1,2013-11-01 03:30:00,82.0947,NaN


### 196. Create activity_growth — the recent window against a prior baseline window.

`activity_growth` compares the recent 24-hour average activity with the previous 24-hour baseline average.

Formula:

`(recent_24h_average - previous_24h_average) / previous_24h_average`

A positive value means activity is increasing, while a negative value means activity is decreasing.


In [4]:
# 196. Create activity_growth — the recent window against a prior baseline window.

df["recent_avg"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.rolling(window=24, min_periods=24).mean())
)

df["previous_avg"] = (
    df.groupby("grid_id")["recent_avg"]
      .transform(lambda x: x.shift(24))
)

df["activity_growth"] = (
    (df["recent_avg"] - df["previous_avg"]) /
    df["previous_avg"].replace(0, pd.NA)
)

In [5]:
# 196. Verify activity_growth for grid 1.

df[df["grid_id"] == 1][
    ["feature_timestamp", "recent_avg", "previous_avg", "activity_growth"]
].tail(20)

,feature_timestamp,recent_avg,previous_avg,activity_growth
135,2013-11-06 20:30:00,66.154583,77.358408,-0.144830
136,2013-11-06 23:30:00,65.581646,75.541750,-0.131849
137,2013-11-07 00:30:00,65.964329,73.702421,-0.104991
138,2013-11-07 01:30:00,67.096967,72.570221,-0.075420
139,2013-11-07 02:30:00,68.178612,72.783658,-0.063270
140,2013-11-07 03:30:00,67.827767,74.191192,-0.085771
141,2013-11-07 04:30:00,67.537617,75.054533,-0.100153
142,2013-11-07 05:30:00,67.228271,75.135638,-0.105241
143,2013-11-07 06:30:00,67.018183,74.838308,-0.104494
144,2013-11-07 07:30:00,66.953542,74.301171,-0.098890


### 197. Create active_hours — the count of hours in the window with activity above zero.

`active_hours` is the number of hours in the recent 24-hour window where `total_activity` is greater than zero.

It represents how consistently active the grid was during the recent window.


In [6]:
# 197. Create active_hours — the count of hours in the window with activity above zero.

df["active_hours"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.gt(0).rolling(window=24, min_periods=24).sum())
)

df["active_hours"] = df["active_hours"].astype("Int64")

In [7]:
# 197. Verify active_hours for grid 1.

df[df["grid_id"] == 1][
    ["feature_timestamp", "total_activity", "active_hours"]
].tail(10)

,feature_timestamp,total_activity,active_hours
145,2013-11-07 08:30:00,80.7752,24
146,2013-11-07 09:30:00,70.6239,24
147,2013-11-07 10:30:00,77.7854,24
148,2013-11-07 11:30:00,85.0444,24
149,2013-11-07 12:30:00,93.5448,24
150,2013-11-07 13:30:00,106.9549,24
151,2013-11-07 14:30:00,113.4819,24
152,2013-11-07 15:30:00,124.7026,24
153,2013-11-07 16:30:00,83.9087,24
154,2013-11-07 17:30:00,73.7006,24


### 198. Create peak_ratio = peak ÷ average over the window.

`peak_ratio` is the maximum `total_activity` divided by the average `total_activity` over the recent 24-hour window.

It shows how large the highest activity level was compared with the grid's recent average.


In [8]:
# 198. Create peak_ratio = peak ÷ average over the window.

df["peak_activity"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.rolling(window=24, min_periods=24).max())
)

df["peak_ratio"] = (
    df["peak_activity"] / df["avg_activity"].replace(0, pd.NA)
)

In [9]:
# 198. Verify peak_ratio for grid 1.

df[df["grid_id"] == 1][
    ["feature_timestamp", "avg_activity", "peak_activity", "peak_ratio"]
].tail(10)

,feature_timestamp,avg_activity,peak_activity,peak_ratio
145,2013-11-07 08:30:00,67.569512,104.5778,1.547707
146,2013-11-07 09:30:00,67.494008,104.5778,1.549438
147,2013-11-07 10:30:00,68.172092,104.5778,1.534027
148,2013-11-07 11:30:00,69.010521,104.5778,1.515389
149,2013-11-07 12:30:00,69.780979,104.5778,1.498658
150,2013-11-07 13:30:00,70.791071,106.9549,1.510853
151,2013-11-07 14:30:00,71.531117,113.4819,1.586469
152,2013-11-07 15:30:00,72.726558,124.7026,1.714678
153,2013-11-07 16:30:00,71.865346,124.7026,1.735226
154,2013-11-07 17:30:00,71.289875,124.7026,1.749233


### 199. Create variability using standard deviation or a coefficient-of-variation style proxy.

`variability` is the standard deviation of `total_activity` over the recent 24-hour trailing window.

It measures how much activity varies during the window. Higher values indicate greater variation.


In [10]:
# 199. Create variability using standard deviation or a coefficient-of-variation style proxy.

df["variability"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.rolling(window=24, min_periods=24).std())
)

In [11]:
# 199. Verify variability for grid 1.

df[df["grid_id"] == 1][
    ["feature_timestamp", "total_activity", "variability"]
].tail(10)

,feature_timestamp,total_activity,variability
145,2013-11-07 08:30:00,80.7752,18.975552
146,2013-11-07 09:30:00,70.6239,18.958944
147,2013-11-07 10:30:00,77.7854,19.026574
148,2013-11-07 11:30:00,85.0444,19.318253
149,2013-11-07 12:30:00,93.5448,19.928837
150,2013-11-07 13:30:00,106.9549,21.187394
151,2013-11-07 14:30:00,113.4819,22.373025
152,2013-11-07 15:30:00,124.7026,24.411595
153,2013-11-07 16:30:00,83.9087,23.589824
154,2013-11-07 17:30:00,73.7006,23.358864


### 200. Create internet_share = internet_activity ÷ total_activity.

`internet_share` is calculated as `internet_activity / total_activity` for the feature timestamp `t`.

It represents the proportion of total activity that comes from internet activity. Division by zero will be handled explicitly.


In [13]:
# 200. Create internet_share = internet_activity ÷ total_activity.

query = """
SELECT
    f.grid_id,
    d.timestamp AS feature_timestamp,
    f.total_activity,
    f.internet_activity
FROM fact_network_activity f
JOIN dim_time d
    ON f.time_key = d.time_key
ORDER BY f.grid_id, d.timestamp;
"""

df = pd.read_sql(query, conn)

df["feature_timestamp"] = pd.to_datetime(df["feature_timestamp"])

df["internet_share"] = (
    df["internet_activity"] /
    df["total_activity"].replace(0, pd.NA)
)

df.head()

D:\NOPIS\tmp\ipykernel_23208\661894847.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,grid_id,feature_timestamp,total_activity,internet_activity,internet_share
0,1,2013-10-31 18:30:00,61.6037,57.7729,0.937815
1,1,2013-10-31 19:30:00,46.1967,44.0469,0.953464
2,1,2013-10-31 20:30:00,42.0324,41.1798,0.979716
3,1,2013-10-31 21:30:00,35.0705,33.0221,0.941592
4,1,2013-10-31 22:30:00,32.2687,31.3769,0.972363


In [14]:
# 200. Verify internet_share for grid 1.

df[df["grid_id"] == 1][
    ["feature_timestamp", "internet_activity", "total_activity", "internet_share"]
].tail(10)

,feature_timestamp,internet_activity,total_activity,internet_share
145,2013-11-07 08:30:00,64.9634,80.7752,0.804249
146,2013-11-07 09:30:00,55.0174,70.6239,0.779020
147,2013-11-07 10:30:00,59.8269,77.7854,0.769128
148,2013-11-07 11:30:00,63.4919,85.0444,0.746574
149,2013-11-07 12:30:00,71.3139,93.5448,0.762350
150,2013-11-07 13:30:00,88.3282,106.9549,0.825845
151,2013-11-07 14:30:00,98.8796,113.4819,0.871325
152,2013-11-07 15:30:00,110.1223,124.7026,0.883079
153,2013-11-07 16:30:00,74.7712,83.9087,0.891102
154,2013-11-07 17:30:00,66.9651,73.7006,0.908610


### 201. Persist the feature table with grid_id and feature_timestamp, where feature_timestamp is t — the last interval the features are allowed to see.

We will persist a feature table containing `grid_id`, `feature_timestamp`, and the six engineered features.

`feature_timestamp` represents time `t`, the last interval that the features are allowed to use. The model will use these features to predict the next interval `t+1`.


In [15]:
# ML2 — Create the complete network activity feature set

query = """
SELECT
    f.grid_id,
    d.timestamp AS feature_timestamp,
    f.total_activity,
    f.internet_activity
FROM fact_network_activity f
JOIN dim_time d
    ON f.time_key = d.time_key
ORDER BY f.grid_id, d.timestamp;
"""

df = pd.read_sql(query, conn)

df["feature_timestamp"] = pd.to_datetime(df["feature_timestamp"])

# 1. Recent 24-hour average
df["avg_activity"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.rolling(24, min_periods=24).mean())
)

# 2. Previous 24-hour baseline
df["previous_avg"] = (
    df.groupby("grid_id")["avg_activity"]
      .transform(lambda x: x.shift(24))
)

# 3. Activity growth
df["activity_growth"] = (
    (df["avg_activity"] - df["previous_avg"]) /
    df["previous_avg"].replace(0, pd.NA)
)

# 4. Active hours in recent 24 hours
df["active_hours"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.gt(0).rolling(24, min_periods=24).sum())
      .astype("Int64")
)

# 5. Peak activity in recent 24 hours
df["peak_activity"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.rolling(24, min_periods=24).max())
)

# 6. Peak ratio
df["peak_ratio"] = (
    df["peak_activity"] /
    df["avg_activity"].replace(0, pd.NA)
)

# 7. Variability
df["variability"] = (
    df.groupby("grid_id")["total_activity"]
      .transform(lambda x: x.rolling(24, min_periods=24).std())
)

# 8. Internet share
df["internet_share"] = (
    df["internet_activity"] /
    df["total_activity"].replace(0, pd.NA)
)

# Keep only the final ML features
feature_df = df[
    [
        "grid_id",
        "feature_timestamp",
        "avg_activity",
        "activity_growth",
        "active_hours",
        "peak_ratio",
        "variability",
        "internet_share"
    ]
].copy()

feature_df.head()

D:\NOPIS\tmp\ipykernel_23208\4061519559.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,grid_id,feature_timestamp,avg_activity,activity_growth,active_hours,peak_ratio,variability,internet_share
0,1,2013-10-31 18:30:00,NaN,NaN,<NA>,NaN,NaN,0.937815
1,1,2013-10-31 19:30:00,NaN,NaN,<NA>,NaN,NaN,0.953464
2,1,2013-10-31 20:30:00,NaN,NaN,<NA>,NaN,NaN,0.979716
3,1,2013-10-31 21:30:00,NaN,NaN,<NA>,NaN,NaN,0.941592
4,1,2013-10-31 22:30:00,NaN,NaN,<NA>,NaN,NaN,0.972363


In [16]:
# ML2 — Check the complete feature set

feature_df[feature_df["grid_id"] == 1].tail(10)

,grid_id,feature_timestamp,avg_activity,activity_growth,active_hours,peak_ratio,variability,internet_share
145,1,2013-11-07 08:30:00,67.569512,-0.085642,24,1.547707,18.975552,0.804249
146,1,2013-11-07 09:30:00,67.494008,-0.087157,24,1.549438,18.958944,0.779020
147,1,2013-11-07 10:30:00,68.172092,-0.071964,24,1.534027,19.026574,0.769128
148,1,2013-11-07 11:30:00,69.010521,-0.057183,24,1.515389,19.318253,0.746574
149,1,2013-11-07 12:30:00,69.780979,-0.050636,24,1.498658,19.928837,0.762350
150,1,2013-11-07 13:30:00,70.791071,-0.042633,24,1.510853,21.187394,0.825845
151,1,2013-11-07 14:30:00,71.531117,-0.040731,24,1.586469,22.373025,0.871325
152,1,2013-11-07 15:30:00,72.726558,-0.029137,24,1.714678,24.411595,0.883079
153,1,2013-11-07 16:30:00,71.865346,-0.043673,24,1.735226,23.589824,0.891102
154,1,2013-11-07 17:30:00,71.289875,-0.044579,24,1.749233,23.358864,0.908610


In [17]:
# 201. Persist the feature table with grid_id and feature_timestamp, where feature_timestamp is t — the last interval the features are allowed to see.

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS network_feature_table (
    grid_id INT NOT NULL,
    feature_timestamp DATETIME NOT NULL,
    avg_activity DOUBLE,
    activity_growth DOUBLE,
    active_hours INT,
    peak_ratio DOUBLE,
    variability DOUBLE,
    internet_share DOUBLE,
    PRIMARY KEY (grid_id, feature_timestamp)
)
""")

conn.commit()

cursor.close()

True

In [24]:
# 201. Persist the feature table with grid_id and feature_timestamp, where feature_timestamp is t — the last interval the features are allowed to see.

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="nopis"
)

print("MySQL connected:", conn.is_connected())

MySQL connected: True


In [25]:
# 201. Persist the feature table with grid_id and feature_timestamp, where feature_timestamp is t — the last interval the features are allowed to see.

feature_df_clean = feature_df.astype(object).where(
    pd.notna(feature_df),
    None
)

data = [
    tuple(row)
    for row in feature_df_clean.itertuples(index=False, name=None)
]

insert_query = """
INSERT INTO network_feature_table (
    grid_id,
    feature_timestamp,
    avg_activity,
    activity_growth,
    active_hours,
    peak_ratio,
    variability,
    internet_share
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    avg_activity = VALUES(avg_activity),
    activity_growth = VALUES(activity_growth),
    active_hours = VALUES(active_hours),
    peak_ratio = VALUES(peak_ratio),
    variability = VALUES(variability),
    internet_share = VALUES(internet_share)
"""

batch_size = 5000

cursor = conn.cursor()

for start in range(0, len(data), batch_size):
    batch = data[start:start + batch_size]
    
    cursor.executemany(insert_query, batch)
    conn.commit()
    
    print(f"Inserted {min(start + batch_size, len(data)):,} / {len(data):,}")

cursor.close()

print("Feature table persisted successfully.")

Inserted 5,000 / 1,556,206
Inserted 10,000 / 1,556,206
Inserted 15,000 / 1,556,206
Inserted 20,000 / 1,556,206
Inserted 25,000 / 1,556,206
Inserted 30,000 / 1,556,206
Inserted 35,000 / 1,556,206
Inserted 40,000 / 1,556,206
Inserted 45,000 / 1,556,206
Inserted 50,000 / 1,556,206
Inserted 55,000 / 1,556,206
Inserted 60,000 / 1,556,206
Inserted 65,000 / 1,556,206
Inserted 70,000 / 1,556,206
Inserted 75,000 / 1,556,206
Inserted 80,000 / 1,556,206
Inserted 85,000 / 1,556,206
Inserted 90,000 / 1,556,206
Inserted 95,000 / 1,556,206
Inserted 100,000 / 1,556,206
Inserted 105,000 / 1,556,206
Inserted 110,000 / 1,556,206
Inserted 115,000 / 1,556,206
Inserted 120,000 / 1,556,206
Inserted 125,000 / 1,556,206
Inserted 130,000 / 1,556,206
Inserted 135,000 / 1,556,206
Inserted 140,000 / 1,556,206
Inserted 145,000 / 1,556,206
Inserted 150,000 / 1,556,206
Inserted 155,000 / 1,556,206
Inserted 160,000 / 1,556,206
Inserted 165,000 / 1,556,206
Inserted 170,000 / 1,556,206
Inserted 175,000 / 1,556,206
Inser

### 202. Write one test that would fail if any feature used data from after feature_timestamp.

We will test that every feature only uses data up to its `feature_timestamp`.

The test will deliberately allow a feature to use data from `t+1`. It must fail when this happens.

This proves that leakage prevention is enforced by a test rather than only being a rule in documentation.


In [26]:
# 202. Write one test that would fail if any feature used data from after feature_timestamp.

def test_no_future_data_used():
    for grid_id in feature_df["grid_id"].unique():
        
        grid_data = df[df["grid_id"] == grid_id].sort_values(
            "feature_timestamp"
        )
        
        for timestamp in grid_data["feature_timestamp"].dropna().unique():
            
            future_data = grid_data[
                grid_data["feature_timestamp"] > timestamp
            ]
            
            # Features at timestamp t must not use future rows
            assert future_data.empty is False or timestamp == grid_data["feature_timestamp"].max()

    print("Leakage boundary test passed.")

In [27]:
# 202. Write one test that would fail if any feature used data from after feature_timestamp.

def test_trailing_window_has_no_future_data():
    
    test_grid = 1
    
    grid_data = df[
        df["grid_id"] == test_grid
    ].sort_values("feature_timestamp").reset_index(drop=True)
    
    for i in range(23, len(grid_data)):
        
        feature_time = grid_data.loc[i, "feature_timestamp"]
        
        # The 24-hour window must end exactly at t
        window = grid_data.iloc[i-23:i+1]
        
        assert window["feature_timestamp"].max() == feature_time
        
        # No row in the window can be after t
        assert (window["feature_timestamp"] <= feature_time).all()
    
    print("Trailing-window leakage test passed.")

In [28]:
test_no_future_data_used()

KeyboardInterrupt: 

In [ ]:
test_trailing_window_has_no_future_data()

In [29]:
# ML2 Validation — Get the 24 underlying hourly rows for the hand-check

grid_id = 1
check_time = pd.Timestamp("2013-11-07 17:30:00")

check_window = df[
    (df["grid_id"] == grid_id) &
    (df["feature_timestamp"] <= check_time)
].sort_values("feature_timestamp").tail(24)

check_window[
    ["feature_timestamp", "total_activity"]
]

,feature_timestamp,total_activity
131,2013-11-06 16:30:00,85.9241
132,2013-11-06 17:30:00,70.1067
133,2013-11-06 18:30:00,59.4471
134,2013-11-06 19:30:00,44.5257
135,2013-11-06 20:30:00,36.9394
136,2013-11-06 23:30:00,30.2773
137,2013-11-07 00:30:00,39.1093
138,2013-11-07 01:30:00,59.8345
139,2013-11-07 02:30:00,70.0247
140,2013-11-07 03:30:00,60.3508


In [30]:
# ML2 Validation — Manually calculate avg_activity and peak_ratio

manual_avg = check_window["total_activity"].mean()
manual_peak = check_window["total_activity"].max()
manual_peak_ratio = manual_peak / manual_avg

print("Manual avg_activity:", manual_avg)
print("Manual peak:", manual_peak)
print("Manual peak_ratio:", manual_peak_ratio)

Manual avg_activity: 71.289875
Manual peak: 124.70259999999999
Manual peak_ratio: 1.7492329731255665


In [31]:
# ML2 Validation — Compare manual calculations with generated features

feature_row = feature_df[
    (feature_df["grid_id"] == grid_id) &
    (feature_df["feature_timestamp"] == check_time)
].iloc[0]

print("Generated avg_activity:", feature_row["avg_activity"])
print("Generated peak_ratio:", feature_row["peak_ratio"])

print(
    "avg_activity matches:",
    abs(manual_avg - feature_row["avg_activity"]) < 1e-10
)

print(
    "peak_ratio matches:",
    abs(manual_peak_ratio - feature_row["peak_ratio"]) < 1e-10
)

Generated avg_activity: 71.289875
Generated peak_ratio: 1.7492329731255665
avg_activity matches: True
peak_ratio matches: True


In [32]:
# 202. Write one test that would fail if any feature used data from after feature_timestamp.

def test_no_future_data_in_feature_window():
    for grid_id in feature_df["grid_id"].unique():
        
        grid_data = df[
            df["grid_id"] == grid_id
        ].sort_values("feature_timestamp").reset_index(drop=True)
        
        for i in range(23, len(grid_data)):
            
            feature_time = grid_data.loc[i, "feature_timestamp"]
            
            # 24-hour trailing window ending at t
            window = grid_data.iloc[i-23:i+1]
            
            # The latest data used must be exactly t
            assert window["feature_timestamp"].max() == feature_time
            
            # Nothing after t can be inside the window
            assert (window["feature_timestamp"] <= feature_time).all()

    print("PASS: No future data is used.")

In [33]:
# 202. Run the leakage test.

test_no_future_data_in_feature_window()

PASS: No future data is used.


In [35]:
# 202. Demonstrate that the leakage test fails when future data is included.

def test_broken_future_window():
    test_grid = 1

    grid_data = df[
        df["grid_id"] == test_grid
    ].sort_values("feature_timestamp").reset_index(drop=True)

    for i in range(23, len(grid_data) - 1):

        feature_time = grid_data.loc[i, "feature_timestamp"]

        # DELIBERATELY WRONG: includes t+1
        broken_window = grid_data.iloc[i-23:i+2]

        # This should fail because t+1 is included
        assert (broken_window["feature_timestamp"] <= feature_time).all()

test_broken_future_window()

AssertionError: 

In [ ]:
# ML2 Validation — Check for undefined or infinite feature values

feature_columns = [
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share"
]

print("Missing values:")
print(feature_df[feature_columns].isna().sum())

print("\nInfinite values:")
print(
    feature_df[feature_columns]
    .apply(lambda col: col.isin([float("inf"), float("-inf")]).sum())
)

Missing values:
avg_activity       229954
activity_growth    469906
active_hours       229954
peak_ratio         229954
variability        229954
internet_share          0
dtype: int64

Infinite values:
avg_activity       0
activity_growth    0
active_hours       0
peak_ratio         0
variability        0
internet_share     0
dtype: int64


: 